# I-EIP Monitor — Tutorial

**Internal Epistemic Invariance Principle monitoring for deployed AI systems.**

This notebook walks through the monitoring half of the I-EIP framework in the order a contributor would learn it:

1. The problem I-EIP addresses
2. Read-only activation **probes** on a live model
3. Estimating the representation map **ρ** via regularized Procrustes
4. Measuring **equivariance error**
5. Running **drift detection** over time
6. Checking **non-degeneracy** of internal representations
7. Assembling the full **IEIPReport**
8. What I-EIP does **NOT** detect (load-bearing — read carefully)
9. **Sprint placeholders** — skeletons for the governance / gating half that students will implement

**Reading (before starting):**

- [`I-EIP_Monitor_Whitepaper`](../I-EIP_Monitor_Whitepaper.md) — the contract
- [`GUASS_SAI.md` §16](../guides/GUASS_SAI.md) — the foundational criterion
- [`I-EIP_Monitor_Sprint_Plan`](../development/I-EIP_Monitor_Sprint_Plan.md) — tasks you can claim

*Andrew H. Bond, April 2026*

## 0. Setup

Install the `[ieip]` extra for PyTorch:

```bash
pip install -e '.[dev,ieip]'
pip install matplotlib  # for the plots in this notebook
```

In [ ]:
from __future__ import annotations

import json
import sys

import numpy as np

# Core monitoring pieces — no torch dependency at top level
from erisml.ieip import (
    AlertLevel,
    DriftDetector,
    DriftReport,
    EquivarianceResult,
    IEIPReport,
    NondegeneracyMetrics,
    ProbeSpec,
    TransformSpec,
    aggregate_report,
    compute_drift_alert,
    effective_rank,
    equivariance_error,
    equivariance_errors_batch,
    estimate_rho,
    nondegeneracy_report,
    reconstruction_error,
)
from erisml.ieip.report import format_json, format_text, max_alert_level
from erisml.ieip.rho import split_pairs

print(f'Python: {sys.version.split()[0]}')
print(f'NumPy: {np.__version__}')

In [ ]:
# PyTorch is optional; gated import so the notebook can still teach the
# numpy-only parts if torch is missing.
try:
    import torch
    from torch import nn

    TORCH_AVAILABLE = True
    print(f'PyTorch: {torch.__version__}')
except ImportError:
    TORCH_AVAILABLE = False
    print('PyTorch not installed. Sections using probes will be skipped.')
    print("Install with: pip install 'erisml-lib[ieip]'")

In [ ]:
# Matplotlib is optional; degrade gracefully.
try:
    import matplotlib.pyplot as plt

    MPL_AVAILABLE = True
except ImportError:
    MPL_AVAILABLE = False
    print('matplotlib not installed; plots will be skipped.')


def _plot_if_available(fn):
    """Decorator: run a plotting function only if matplotlib is available."""

    def wrapper(*args, **kwargs):
        if not MPL_AVAILABLE:
            print(f'(skipping plot: matplotlib not installed)')
            return None
        return fn(*args, **kwargs)

    return wrapper

## 1. The Problem

The **Epistemic Invariance Principle (EIP)** says a model's judgment should be invariant under meaning-preserving transformations of its input: paraphrase the question, swap variable names, change units — the answer shouldn't change.

The **Internal** EIP (§16 of GUASS-SAI) says something strictly stronger:

$$h_\ell(g \cdot x) \;\approx\; \rho_\ell(g) \cdot h_\ell(x)$$

— the *internal representations* at every probed layer should transform consistently when the input is transformed, via some (unknown but estimable) representation map `ρ_ℓ(g)`.

Why internal? Because a model whose *outputs* look invariant but whose *activations* drift wildly under meaning-preserving transforms is exhibiting one of three failure modes the output alone cannot distinguish:

| Failure mode | Signature |
|---|---|
| **Spec-gaming** | Outputs look fine; internal state depends heavily on surface features of the prompt |
| **Drift** | Equivariance error grows slowly across updates / fine-tunes |
| **Degeneracy** | Activations collapse to a low-rank subspace; many inputs produce identical internal state |

The monitoring half of I-EIP gives us tools to detect these. Detection is a prerequisite for the **governance half** students will build out in the sprint plan.

## 2. A Toy Model

Instead of pulling a HuggingFace transformer (slow, requires network), we use a tiny MLP. It is small enough to reason about and big enough that representations are non-trivial.

In [ ]:
if TORCH_AVAILABLE:
    torch.manual_seed(0)

    model = nn.Sequential(
        nn.Linear(16, 64),
        nn.Tanh(),
        nn.Linear(64, 64),
        nn.Tanh(),
        nn.Linear(64, 8),
    )
    model.eval()  # we will not train in this notebook
    print(model)
else:
    print('(torch not available — building a numpy stand-in for later cells)')

### Picking a transformation `g`

For a toy demo we use a scalar input transformation: `g(x) = 1.05 · x`. On a well-conditioned MLP with smooth activations this is approximately equivariant at the hidden layer — there should exist a matrix `ρ` that predicts `h(g·x)` from `h(x)` with low error.

In real deployments `g` would be something meaning-preserving: paraphrase, name-swap, unit-change, order-permutation. The math is identical; only the data-collection step differs.

## 3. Activation Probes

An `ActivationProbe` wraps `torch.nn.Module.register_forward_hook` with three guarantees:

1. **Read-only** — hooks return `None`, so the forward pass is bit-equivalent with or without probes attached
2. **Fail-closed** — on probe-internal failures, the probe stops recording; downstream code is expected to treat this as veto-worthy
3. **Shape-validated** — if `expected_shape` is set, mismatches are failures

Create a probe, attach it to a module, run the model, then detach.

In [ ]:
if TORCH_AVAILABLE:
    from erisml.ieip import ActivationProbe, ProbeManager

    # Probe the second Tanh (index 3 in the Sequential).
    probe_spec = ProbeSpec(
        target_layer=3,
        activation_site='residual',
        name='tanh_mid',
        expected_shape=(64,),
    )
    probe = ActivationProbe(module=model[3], spec=probe_spec)

    probe.attach()
    with torch.no_grad():
        _ = model(torch.randn(32, 16))
    probe.detach()

    acts = probe.collected()
    print(f'Captured activations shape: {acts.shape}')  # (32, 64)
    print(f'Forward-pass calls observed: {probe.call_count}')
    print(f'Failures: {probe.failure_count}')

### Read-only invariant

Critical property: **attaching a probe must not change the model's outputs.** This test is also in the unit-test suite (`tests/ieip/test_probes.py::test_does_not_modify_forward_output`); we repeat it here to build intuition.

In [ ]:
if TORCH_AVAILABLE:
    x = torch.randn(8, 16)

    with torch.no_grad():
        y_without = model(x).clone()

    probe = ActivationProbe(model[3], ProbeSpec(target_layer=3, name='test'))
    probe.attach()
    with torch.no_grad():
        y_with = model(x)
    probe.detach()

    assert torch.equal(y_without, y_with), 'probe modified the output!'
    print('Read-only invariant holds: outputs are bitwise identical.')

### Managing multiple probes

`ProbeManager` is a context-managed collection. Useful when probing several layers at once.

In [ ]:
if TORCH_AVAILABLE:
    mgr = ProbeManager()
    mgr.add(model[1], ProbeSpec(target_layer=1, name='tanh_early'))
    mgr.add(model[3], ProbeSpec(target_layer=3, name='tanh_mid'))

    with mgr.active():
        with torch.no_grad():
            _ = model(torch.randn(16, 16))

    acts_per_probe = mgr.collected()
    for name, a in acts_per_probe.items():
        print(f'{name}: shape {a.shape}, mean={a.mean():.4f}, std={a.std():.4f}')

## 4. Estimating ρ via Regularized Procrustes

Given paired activations `(h(x), h(g·x))` from a calibration corpus, we want the matrix `ρ` that best predicts the second from the first:

$$\hat\rho = \arg\min_\rho \|Y - X\rho^T\|_F^2 + \lambda \|\rho\|_F^2$$

Closed form:

$$\hat\rho = Y^T X (X^T X + \lambda I)^{-1}$$

Ridge regularization stabilizes the solve and prevents overfitting on the calibration set.

We test the estimator with a **known** linear map to verify it works: let `ρ_true` be a random orthogonal matrix, let `Y = X @ ρ_true^T`, then confirm `estimate_rho(X, Y)` recovers `ρ_true` up to the regularization bias.

In [ ]:
rng = np.random.default_rng(42)
n, d = 500, 16

X = rng.standard_normal(size=(n, d))
# Known ground-truth ρ: random orthogonal matrix
rho_true, _ = np.linalg.qr(rng.standard_normal(size=(d, d)))
Y = X @ rho_true.T

rho_hat = estimate_rho(X, Y, lambda_reg=1e-8)
err = reconstruction_error(X, Y, rho_hat, relative=True)
diff = np.linalg.norm(rho_hat - rho_true) / np.linalg.norm(rho_true)

print(f'Relative reconstruction error: {err:.2e}')
print(f'Relative difference from ρ_true: {diff:.2e}')

### The regularization trade-off

On noise-only pairs `(X, Y)` with no real relationship, ridge should drive ρ toward zero. Plot ‖ρ‖ against λ to see this:

In [ ]:
@_plot_if_available
def plot_regularization_curve():
    rng = np.random.default_rng(0)
    X_noise = rng.standard_normal(size=(100, 16))
    Y_noise = rng.standard_normal(size=(100, 16))

    lambdas = np.logspace(-8, 2, 30)
    norms = [np.linalg.norm(estimate_rho(X_noise, Y_noise, lambda_reg=lam)) for lam in lambdas]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.semilogx(lambdas, norms, marker='o')
    ax.set_xlabel('λ (regularization)')
    ax.set_ylabel('‖ρ̂‖ (Frobenius)')
    ax.set_title('Regularization shrinks ρ toward 0 on pure-noise pairs')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_regularization_curve()

### Train/val splits

For rigorous calibration, hold out a validation set. `split_pairs` gives a reproducible shuffle and split.

In [ ]:
X_all = rng.standard_normal(size=(400, 16))
Y_all = X_all @ rho_true.T + 0.01 * rng.standard_normal(size=(400, 16))

X_tr, Y_tr, X_val, Y_val = split_pairs(X_all, Y_all, val_fraction=0.2, seed=0)

rho_tr = estimate_rho(X_tr, Y_tr, lambda_reg=1e-4)
train_err = reconstruction_error(X_tr, Y_tr, rho_tr)
val_err = reconstruction_error(X_val, Y_val, rho_tr)

print(f'Train error: {train_err:.4f}')
print(f'Val error:   {val_err:.4f}')
print('Large gap between train and val → overfitting; tune λ.')

## 5. Equivariance Error on the Live Model

Now combine probes + ρ estimation to measure equivariance on the toy model for the transformation `g(x) = 1.05 · x`.

**Protocol:** for a batch of inputs `x`, collect `h(x)` by running the model on `x` and `h(g·x)` by running on `1.05·x` with the same probe. Then estimate ρ from a calibration portion, and evaluate on a held-out portion.

In [ ]:
if TORCH_AVAILABLE:

    def collect_pair(model, probe, inputs, scale):
        probe.clear()
        probe.attach()
        with torch.no_grad():
            _ = model(inputs)
        H_x = probe.collected().copy()
        probe.clear()
        with torch.no_grad():
            _ = model(scale * inputs)
        H_gx = probe.collected().copy()
        probe.detach()
        return H_x, H_gx

    probe = ActivationProbe(model[3], ProbeSpec(target_layer=3, name='mid'))
    rng_torch = torch.Generator().manual_seed(7)

    # Calibration batch
    x_cal = torch.randn((256, 16), generator=rng_torch)
    H_cal, G_cal = collect_pair(model, probe, x_cal, scale=1.05)

    rho = estimate_rho(H_cal, G_cal, lambda_reg=1e-4)

    # Live evaluation batch
    x_live = torch.randn((64, 16), generator=rng_torch)
    H_live, G_live = collect_pair(model, probe, x_live, scale=1.05)

    result = equivariance_error(H_live, G_live, rho, layer=3, transform='scale_1.05')
    print(result)

### Why is the error small?

The tanh network with scaled input `1.05 · x` produces approximately-scaled hidden activations for small scale factors, because `tanh` is approximately linear near the origin. A learned `ρ` captures that linear relationship with low residual.

Try larger scales (e.g., `scale=2.0`) and the error grows — `tanh` saturates, and the transformation stops being approximately-linear. This is a meaningful diagnostic: equivariance is *a claim about a specific (model, transform, layer) triple*, not a global property.

In [ ]:
if TORCH_AVAILABLE:
    scales = [1.01, 1.05, 1.25, 1.5, 2.0, 3.0, 5.0]
    errs = []
    for s in scales:
        H_s, G_s = collect_pair(model, probe, x_cal, scale=s)
        rho_s = estimate_rho(H_s, G_s, lambda_reg=1e-4)
        H_live_s, G_live_s = collect_pair(model, probe, x_live, scale=s)
        r = equivariance_error(H_live_s, G_live_s, rho_s, layer=3, transform=f'scale_{s}')
        errs.append(r.error)

    for s, e in zip(scales, errs):
        print(f'scale={s:<5}  error={e:.4f}')

    @_plot_if_available
    def plot_scale_errors():
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(scales, errs, marker='o')
        ax.set_xlabel('scale factor (transformation strength)')
        ax.set_ylabel('Relative equivariance error')
        ax.set_title('Equivariance breaks down as tanh saturates')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    plot_scale_errors()

## 6. Drift Detection

A deployed model's equivariance error should be stable across inferences. **Drift** is a sustained departure from baseline — e.g., due to accumulated fine-tuning, input-distribution shift, or adversarial prefix buildup.

`DriftDetector` maintains an EWMA baseline per `(layer, transform)` key:

$$\text{baseline}_{t+1} = \alpha \cdot \text{error}_t + (1 - \alpha) \cdot \text{baseline}_t$$

and raises an alert when the new observation departs from the baseline by more than configured thresholds.

In [ ]:
# Simulate 100 inferences worth of equivariance errors.
# Scenario: stable for 60 steps, then gradual upward drift, then a sudden spike.
rng = np.random.default_rng(2026)

stable = rng.normal(loc=0.02, scale=0.005, size=60)
drifting = rng.normal(loc=0.02, scale=0.005, size=30) + np.linspace(0, 0.04, 30)
spike = rng.normal(loc=0.12, scale=0.01, size=10)
stream = np.concatenate([stable, drifting, spike])
stream = np.clip(stream, 0, None)

detector = DriftDetector(alpha=0.1, warmup_observations=5)
baselines = []
alerts = []
for err in stream:
    rep = detector.observe(layer=3, transform='scale_1.05', error=float(err))
    baselines.append(rep.baseline_error)
    alerts.append(rep.alert_level.value)

# Count alerts
from collections import Counter

print(Counter(alerts))

In [ ]:
@_plot_if_available
def plot_drift_stream():
    fig, ax = plt.subplots(figsize=(10, 4))
    ts = np.arange(len(stream))
    ax.plot(ts, stream, label='equivariance error', color='tab:blue', alpha=0.6)
    ax.plot(ts, baselines, label='EWMA baseline', color='tab:orange', lw=2)

    colors = {'normal': 'tab:green', 'elevated': 'tab:orange', 'critical': 'tab:red'}
    for level in ('elevated', 'critical'):
        indices = [i for i, a in enumerate(alerts) if a == level]
        if indices:
            ax.scatter(
                indices,
                stream[indices],
                color=colors[level],
                label=f'{level} alert',
                zorder=3,
                s=40,
            )

    ax.set_xlabel('inference #')
    ax.set_ylabel('relative equivariance error')
    ax.set_title('Drift detection over a simulated deployment timeline')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_drift_stream()

### Tuning alpha

- `alpha` close to 1 → baseline tracks the current observation (very adaptive, noisy)
- `alpha` close to 0 → baseline barely moves (very stable, slow to adapt)

Deployments with high natural variability want low `alpha` (so short-lived spikes don't overwrite the baseline); deployments with rapid legitimate changes want higher `alpha`.

## 7. Non-degeneracy

A model with **perfect** equivariance but whose activations collapse to a constant or low-rank subspace is also a failure mode — everything looks the same to the monitor, so even drastically misaligned inputs pass the equivariance check.

**Effective rank** (exponential of the Shannon entropy of normalized singular values) is a continuous generalization of matrix rank that ranges in `[1, min(n, d)]`:

$$\text{eff\_rank}(X) = \exp\!\left(-\sum_i p_i \log p_i\right), \quad p_i = \frac{s_i}{\sum_j s_j}$$

Compare a healthy activation matrix to a rank-1 collapse.

In [ ]:
rng = np.random.default_rng(2026)

# Healthy: isotropic Gaussian in 16D
X_healthy = rng.standard_normal(size=(500, 16))
report_healthy = nondegeneracy_report(X_healthy, layer=0)
print('Healthy:')
print(f'  effective rank: {report_healthy.effective_rank:.2f}')
print(f'  alert level:    {report_healthy.alert_level.value}')

# Collapsed: rank 1 — all rows are scaled copies of one direction
direction = rng.standard_normal(size=16)
scales = rng.standard_normal(size=500)
X_collapsed = scales[:, None] * direction[None, :]
report_collapsed = nondegeneracy_report(X_collapsed, layer=0)
print('\nCollapsed:')
print(f'  effective rank: {report_collapsed.effective_rank:.2f}')
print(f'  alert level:    {report_collapsed.alert_level.value}')

# Partially collapsed: dominant 3-dim subspace + noise
basis = rng.standard_normal(size=(16, 3))
codes = rng.standard_normal(size=(500, 3))
X_partial = codes @ basis.T + 0.01 * rng.standard_normal(size=(500, 16))
report_partial = nondegeneracy_report(X_partial, layer=0)
print('\nPartial collapse (rank ~3):')
print(f'  effective rank: {report_partial.effective_rank:.2f}')
print(f'  alert level:    {report_partial.alert_level.value}')

In [ ]:
@_plot_if_available
def plot_rank_sweep():
    target_ranks = range(1, 17)
    eff_ranks = []
    for k in target_ranks:
        # Construct a matrix with k dominant components
        basis_k = rng.standard_normal(size=(16, k))
        codes_k = rng.standard_normal(size=(500, k))
        Xk = codes_k @ basis_k.T
        eff_ranks.append(effective_rank(Xk))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(list(target_ranks), eff_ranks, marker='o', label='measured effective rank')
    ax.plot(list(target_ranks), list(target_ranks), '--', color='gray', label='target')
    ax.set_xlabel('constructed matrix rank k')
    ax.set_ylabel('measured effective rank')
    ax.set_title('Effective rank tracks true rank')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_rank_sweep()

## 8. The Full IEIPReport

All three signals compose into a single report:

In [ ]:
if TORCH_AVAILABLE:
    # Gather a small stream of observations on the live model.
    equiv_results = []
    drift_reports = []
    detector2 = DriftDetector(warmup_observations=3)

    for _ in range(20):
        x_batch = torch.randn((64, 16))
        H_b, G_b = collect_pair(model, probe, x_batch, scale=1.05)
        rho_b = estimate_rho(H_cal, G_cal, lambda_reg=1e-4)  # reuse calibration ρ
        eq = equivariance_error(H_b, G_b, rho_b, layer=3, transform='scale_1.05')
        equiv_results.append(eq)
        drift_reports.append(
            detector2.observe(layer=eq.layer, transform=eq.transform, error=eq.error)
        )

    nondeg = nondegeneracy_report(H_b, layer=3)

    report = aggregate_report(
        equivariance=equiv_results[-3:],
        drift=drift_reports[-3:],
        nondegeneracy=[nondeg],
    )

    print(format_text(report))

In [ ]:
if TORCH_AVAILABLE:
    # JSON form — this is what gets handed to the gating layer and
    # eventually sealed into a cryptographic audit artifact.
    print(format_json(report, indent=2)[:1200])

## 9. What I-EIP Does NOT Detect

This section is **load-bearing**. Read the whitepaper §7 for the full list. Three items to keep in mind when you use this:

1. **Deceptive alignment with consistent internal representations.** A model that has learned to produce aligned-looking activations while pursuing misaligned goals will look fine to the monitor. Equivariance holds; nothing flags.
2. **Strategic equivariance preservation.** A capable model that knows it is monitored and modulates its internal state to pass ρ checks cannot be caught by this framework alone.
3. **Novel transformations outside the registry.** Only declared `TransformSpec`s are checked. An attacker exploiting a transformation not in the registry is invisible.

Illustrate #1 with a thought experiment: suppose we *manually* produce a (ρ, X, Y) triple where the equivariance error is near zero but the activations encode something adversarial. The monitor will report `normal`.

In [ ]:
# Adversarial scenario: X and Y satisfy Y = X @ ρ^T exactly by construction,
# but the 'activations' encode an adversarial-looking pattern. The monitor
# sees perfect equivariance.
rng = np.random.default_rng(99)
d = 32
rho_adv, _ = np.linalg.qr(rng.standard_normal(size=(d, d)))
X_adv = rng.standard_normal(size=(200, d))  # could be anything encoded
Y_adv = X_adv @ rho_adv.T

result_adv = equivariance_error(
    X_adv, Y_adv, rho_adv, layer=0, transform='adversarial'
)
nondeg_adv = nondegeneracy_report(X_adv, layer=0)

print(f'Equivariance error: {result_adv.error:.2e}')
print(f'Effective rank: {nondeg_adv.effective_rank:.2f}')
print(f'Alert: {nondeg_adv.alert_level.value}')
print()
print('Monitor says: healthy. Reality: the activations could encode anything.')
print('This is exactly the failure mode §7 of the whitepaper warns about.')

This is why I-EIP is **one layer of defense in depth**, not a standalone alignment solution. The governance half — EM-DAG gating under a democratically-authored DEME profile — is needed for coverage of behavioral intent, and capability bounds / corrigibility are needed for everything below that.

---

## 10. Sprint Placeholders

The cells below are **starter skeletons** for the open sprint tasks. Each one corresponds to a specific task in the [I-EIP Monitor Sprint Plan](../development/I-EIP_Monitor_Sprint_Plan.md). Claim the task on its GitHub issue before starting.

Each placeholder follows the same pattern:

1. **Why this task matters** — the one-paragraph motivation
2. **References** — whitepaper sections and related modules
3. **Starter code** — a `# TODO:` skeleton you fill in
4. **Acceptance criteria** — a small assertion or test so you know when you're done

### Placeholder 1: EthicalFacts (Sprint B.1.1)

**Why:** Every gating decision requires an `EthicalFacts` record that packages the input hash, probe readings, domain-router output, and proposed continuation into a form the EMs can evaluate.

**References:**

- Whitepaper §4.1 (Per-Inference Pipeline), §5 (Cryptographic Attestation)
- `src/erisml/ethics/facts.py` — existing `EthicalFacts` class for DEME (your type is distinct but similar in spirit)

**Acceptance:** fields are typed; `to_dict()` is JSON-serializable; includes SHA-256 input hash.

In [ ]:
# Sprint B.1.1 — EthicalFacts for I-EIP gating
import hashlib
from dataclasses import dataclass, field


@dataclass
class IEIPEthicalFacts:
    '''Package of signals from one inference for EM consumption.

    TODO (student):
    - Add fields: input_hash (str), domain_label (str), probe_readings
      (dict[str, np.ndarray] — but store hashes not raw tensors for
      audit purposes), proposed_continuation (str), ieip_report
      (IEIPReport), timestamp (float).
    - Implement from_inference(model_input, probe_manager, report, ...)
    - Implement to_dict() and to_json() that are safe to log
      (do NOT log raw activations in production).
    '''

    input_hash: str = ''
    # TODO: add remaining fields per the docstring above

    def to_dict(self) -> dict:
        raise NotImplementedError('student task: Sprint B.1.1')


# Acceptance example — should pass once you implement:
#   facts = IEIPEthicalFacts.from_inference(...)
#   assert len(facts.input_hash) == 64  # SHA-256 hex
#   assert 'ieip_report' in facts.to_dict()
#   assert isinstance(json.dumps(facts.to_dict()), str)

### Placeholder 2: Nullifier Library (Sprint A.4.3)

**Why:** Nullifiers (abuse, danger, impossibility, illegality, estrangement) are absorbing states in the EM-DAG. They have cross-domain priority and trigger immediate veto regardless of which domain module is active.

**References:**

- Whitepaper §3.2 L2 (Nullifiers)
- `non-abelian-sqnd/docs/dear_abby_em_dag_documentation.md` — empirical nullifier counts from the Dear Abby corpus

**Acceptance:** each nullifier is a callable that maps an `IEIPEthicalFacts` to a bool; registry iterable; tests cover each nullifier's canonical trigger.

In [ ]:
# Sprint A.4.3 — Nullifier library
from typing import Callable

Nullifier = Callable[['IEIPEthicalFacts'], bool]


class NullifierRegistry:
    '''Registry of nullifiers with cross-domain priority.

    TODO (student):
    - Implement add(name, predicate) with duplicate-name guard.
    - Implement evaluate(facts) -> list[str] returning names of all
      nullifiers that triggered.
    - Populate with the canonical five (abuse, danger, impossibility,
      illegality, estrangement). For text-input deployments, these
      will initially be keyword-based; improve to classifiers in a
      follow-up sprint.
    '''

    def __init__(self) -> None:
        self._nullifiers: dict[str, Nullifier] = {}

    def add(self, name: str, predicate: Nullifier) -> None:
        raise NotImplementedError('student task: Sprint A.4.3')

    def evaluate(self, facts: 'IEIPEthicalFacts') -> list[str]:
        raise NotImplementedError('student task: Sprint A.4.3')


# Acceptance:
#   reg = NullifierRegistry()
#   # populate canonical nullifiers
#   triggered = reg.evaluate(facts_with_abuse_keyword)
#   assert 'abuse' in triggered

### Placeholder 3: Domain Discovery (Sprint A.3)

**Why:** The EM-DAG must be **per-model**, not a generic corpus DAG. Domain discovery identifies the model's own concept clusters from activation patterns, so EMs can be grounded in the model's actual representational space.

**References:**

- Whitepaper §3.3 Step 3 (Domain Discovery)
- Mechanistic Correlates Proposal (Bond & Claude 2026), §3 for probing methodology

**Acceptance:** given calibration activations, returns a list of named clusters with representative inputs and confidence scores. Low-confidence clusters are flagged for human review.

In [ ]:
# Sprint A.3 — Domain Discovery
from dataclasses import dataclass
from typing import List


@dataclass
class DiscoveredDomain:
    '''A candidate domain found in the model's activation space.'''

    name: str
    cluster_id: int
    top_exemplars: List[str]
    confidence: float
    needs_human_review: bool = False


def discover_domains(
    activations: np.ndarray,
    exemplar_inputs: List[str],
    n_clusters: int = 10,
    confidence_threshold: float = 0.5,
) -> List[DiscoveredDomain]:
    '''Cluster activations and extract domain labels.

    TODO (student):
    - Use sklearn KMeans (or HDBSCAN if you prefer) on PCA-reduced
      activations.
    - For each cluster, find the top-k inputs closest to the centroid.
    - Compute a confidence metric (silhouette score, within-cluster
      density, or similar).
    - Mark clusters below the threshold for human review.
    - Return a list of DiscoveredDomain.

    For SAE-based extraction (A.3.2), this function wraps an existing
    SAE and extracts features instead of k-means clusters.
    '''
    raise NotImplementedError('student task: Sprint A.3')


# Acceptance:
#   domains = discover_domains(activations=acts, exemplar_inputs=prompts)
#   assert all(isinstance(d, DiscoveredDomain) for d in domains)
#   assert any(d.needs_human_review for d in domains) or all(d.confidence > 0.5 for d in domains)

### Placeholder 4: Governance Aggregation (Sprint B.2.3)

**Why:** Multiple EMs produce independent `EthicalJudgment`s per inference. The governance layer aggregates them under the active DEME profile — stakeholder weights, dimension weights, lexical priorities, hard vetoes — and emits the final gating decision.

**References:**

- Whitepaper §4.1 step 5 (Governance aggregation)
- `src/erisml/ethics/governance/` — existing DEME governance to build on

**Acceptance:** pure function; preserves hard vetoes; respects lexical priorities (a rights-category veto overrides any utility preference); produces a machine-readable rationale listing drivers.

In [ ]:
# Sprint B.2.3 — Governance aggregation
from enum import Enum


class GatingDecision(str, Enum):
    ALLOW = 'allow'
    VETO = 'veto'
    REDIRECT = 'redirect'


@dataclass
class DecisionOutcome:
    decision: GatingDecision
    drivers: List[str]  # EM names / nullifier names that drove it
    lexical_overrides: List[str]
    rationale: str


def aggregate_governance(
    ethical_judgments: list,  # list of EthicalJudgment (define your own type)
    deme_profile: dict,  # stakeholder weights, lexical priorities, hard vetoes
) -> DecisionOutcome:
    '''Aggregate EM judgments into a single gating decision.

    TODO (student):
    - First check hard vetoes (nullifiers + declared hard vetoes) — if
      any trigger, return VETO immediately with drivers populated.
    - Then apply lexical priority layers in order (e.g., rights before
      welfare): if a higher layer has a definitive verdict, lower
      layers cannot override.
    - Within a layer, compute weighted scores per stakeholder and
      aggregate.
    - Default to ALLOW when no vetoes, all priorities satisfied.
    '''
    raise NotImplementedError('student task: Sprint B.2.3')


# Acceptance:
#   outcome = aggregate_governance(judgments, profile)
#   # A hard-veto EM should produce VETO regardless of the others
#   assert outcome.decision == GatingDecision.VETO if any_hard_veto else True

### Placeholder 5: Audit Artifact (Sprint C.1.1)

**Why:** Every gating decision must produce a signed, hash-chained artifact that a third-party regulator can verify offline. The JSON schema is specified in Whitepaper §5.

**References:**

- Whitepaper §5 (Cryptographic Attestation)
- USPTO Provisional Patent 5 (attestation pipeline), December 2025

**Acceptance:** produces a well-formed artifact matching the schema; round-trip JSON (serialize → parse → compare) is stable; includes all fields needed for third-party verification.

In [ ]:
# Sprint C.1.1 — Audit artifact schema and builder
@dataclass
class AuditArtifact:
    '''Per-inference audit artifact.

    TODO (student):
    - Populate fields per Whitepaper §5 schema:
      artifact_version, timestamp_utc, deployment_id, model_fingerprint,
      input_hash, probe_readings (hashes, not raw tensors),
      ieip_report (from IEIPReport.to_dict()), em_dag_version,
      deme_profile_version, ethical_judgments (summarized),
      governance_decision, signature (filled in at Sprint C.1.4 by
      the ECDSA signer — leave as placeholder string for this task).
    - Implement to_json() that is canonical (sorted keys) so the hash
      is stable across runs.
    '''

    artifact_version: str = '1.0'
    # TODO: add the remaining fields

    def to_json(self) -> str:
        raise NotImplementedError('student task: Sprint C.1.1')

    def content_hash(self) -> str:
        raise NotImplementedError(
            'student task: Sprint C.1.3 (sha256 of canonical JSON)'
        )


# Acceptance:
#   art = AuditArtifact(...)
#   s1, s2 = art.to_json(), art.to_json()
#   assert s1 == s2  # canonical and deterministic
#   parsed = json.loads(s1)
#   assert parsed['artifact_version'] == '1.0'

### Placeholder 6: GatedForward Wrapper (Sprint B.3.1)

**Why:** The end-to-end integration. Wraps a `torch.nn.Module` so that every forward call runs the probe → ρ check → EM evaluation → governance → gating pipeline, and returns either the model's output (allow), a safe default (veto), or a redirect continuation.

**References:**

- Whitepaper §2.1 (architecture diagram), §4 (runtime gating)
- Modules from this notebook: `ProbeManager`, `DriftDetector`, the placeholders above

**Acceptance:** allow path is bit-equivalent to the bare model; veto path emits the declared safe default; every forward produces an `AuditArtifact`.

In [ ]:
# Sprint B.3.1 — GatedForward integration
if TORCH_AVAILABLE:

    class GatedForward(nn.Module):
        '''Wrap a model so each forward call runs the full I-EIP pipeline.

        TODO (student):
        - Accept (inner_model, probe_manager, rho_map, em_dag, deme_profile,
          safe_default, redirect_map, attestation_sink) at construction.
        - On forward(x):
            1. Run the inner model while probes are attached (use
               probe_manager.active()).
            2. Compute equivariance error(s) with the precomputed ρ map.
            3. Observe drift with the detector.
            4. Compute non-degeneracy on the batch.
            5. Assemble IEIPReport.
            6. Construct IEIPEthicalFacts and evaluate EMs.
            7. Aggregate via aggregate_governance.
            8. Build AuditArtifact and send to attestation_sink.
            9. Return the model output if ALLOW, safe_default if VETO,
               redirect_map[driver] if REDIRECT.
        - Keep the out-of-model trust boundary: the gating code must not
          leak timing or memory access patterns back to the inner model.
        '''

        def __init__(self, inner_model: nn.Module, **kwargs):
            super().__init__()
            self.inner = inner_model
            # TODO: store the other arguments

        def forward(self, x):
            raise NotImplementedError('student task: Sprint B.3.1')


# Acceptance:
#   gated = GatedForward(model, probe_manager=mgr, rho_map=rho, ...)
#   # On benign inputs, gated(x) ≈ model(x)
#   # On inputs that trigger a nullifier, gated(x) == safe_default

## 11. Where to Go Next

You now have:

- A working `estimate_rho` / `equivariance_error` / `DriftDetector` / `nondegeneracy_report` / `IEIPReport` pipeline
- Read-only PyTorch probes that do not perturb the model
- Six concrete sprint-placeholder cells covering the key gaps

**To contribute:**

1. Read [`I-EIP_Monitor_Sprint_Plan`](../development/I-EIP_Monitor_Sprint_Plan.md) §Getting Started.
2. Pick a task from the First-Week Starter Tasks table.
3. Claim it by commenting on its tracking issue.
4. Build from the starter skeleton here or under `src/erisml/ieip/` if the task is production.
5. Add tests in `tests/ieip/` following the patterns already there.
6. Open a PR; reference the task ID.

**Guardrails** (see sprint plan for the full list):

- Fail-closed, always.
- The model must not observe gating.
- EM-DAG is per-model, not shared.
- Never soften the 'What I-EIP Does NOT Detect' section.

Good luck.

---

*Questions: andrew.bond@sjsu.edu — or open a discussion on the erisml-lib repo.*